In [0]:
%sql
USE CATALOG workspace;
USE DATABASE batch;

In [0]:
from pyspark.sql.functions import coalesce, lit ,col, dense_rank
from pyspark.sql.window import Window

window_spec = Window.partitionBy("customer_id").orderBy(col("updated_time").desc())

def handle_null(df, columns):
    df = df.withColumns({col: coalesce(df[col], lit(-1)) for col in columns})
    return df

In [0]:
df_customer = spark.read.table("customer_bronzelayer")
df_customer = df_customer.withColumn("rank", dense_rank().over(window_spec)).filter("rank=1").drop("rank")

df_trasaction = spark.sql("select * from (select *,dense_rank() over(partition by transaction_id order by updated_time desc) as rank from transactions_bronzelayer) a where rank=1")

df_trasaction = handle_null(df_trasaction, ["product_id","transaction_id"])

df_product = spark.sql("select * from (select *,dense_rank() over(partition by product_id order by updated_time desc) as rank from product_bronzelayer) a where rank=1")

In [0]:
df_joined = (df_trasaction.alias("t")
    .join(df_customer.alias("c"), col("t.customer_id") == col("c.customer_id"), "left")
    .join(df_product.alias("p"), col("t.product_id") == col("p.product_id"), "left")
    .select("t.transaction_id","t.updated_time","c.name","p.product_name","t.total_amount","t.transaction_date","c.country","c.city","t.status"))



In [0]:
df_joined.write.mode("overwrite").format("delta").saveAsTable("batch_silverlayer")